In [1]:
import random
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim.lr_scheduler as lr_scheduler

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.use_deterministic_algorithms(True)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


In [4]:
mean=(0.4914, 0.4822, 0.4465)
std=(0.2023, 0.1994, 0.2010)

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

In [5]:
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform_train, download=True)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)


In [6]:
def get_dataloaders(seed=42):
    set_seed(seed)
    g = torch.Generator()
    g.manual_seed(seed)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=4, worker_init_fn=seed_worker, generator=g, pin_memory=True)
    testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
    return trainloader, testloader

In [7]:
def vanilla_train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

In [8]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

In [9]:
def train_val_loop(model, optimizer, epochs, training_function, lr_scheduler, print_stats=True, **kwargs):
  trainloader, testloader = get_dataloaders(42)
  criterion = nn.CrossEntropyLoss()
  scaler = torch.amp.GradScaler('cuda')
  if print_stats:
    print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>8}  {'Val Acc':>7}")
    print("-" * 52)
  for epoch in range(epochs):
      current_lr = optimizer.param_groups[0]['lr']
      tr_loss, tr_acc = training_function(model, trainloader, optimizer, criterion, scaler, **kwargs)
      vl_loss, vl_acc = evaluate(model, testloader, criterion)
      if print_stats:
        print(f"{epoch+1:>5}  {tr_loss:>10.4f}  {tr_acc:>9.4f}  {vl_loss:>8.4f}  {vl_acc:>7.4f}")
      lr_scheduler.step()


In [10]:
# num_epochs = 3

# set_seed(42)
# teacher = torch.hub.load('pytorch/vision:v0.10.0', 'resnet50', weights='ResNet50_Weights.IMAGENET1K_V1')
# teacher.fc = nn.Linear(teacher.fc.in_features, 10)
# teacher.to(device)
# teacher_optim = torch.optim.Adam(teacher.parameters(), lr=1e-3)
# teacher_scheduler = lr_scheduler.CosineAnnealingLR(teacher_optim, T_max=num_epochs, eta_min=1e-6)

# train_val_loop(teacher, teacher_optim, num_epochs, vanilla_train_epoch, teacher_scheduler, print_stats=True)

In [11]:
num_epochs = 5

set_seed(42)
teacher = torch.hub.load('pytorch/vision:v0.10.0', 'resnet50', weights='ResNet50_Weights.IMAGENET1K_V1')
teacher.fc = nn.Linear(teacher.fc.in_features, 10)
teacher.to(device)
teacher_optim = torch.optim.Adam(teacher.parameters(), lr=1e-3)
teacher_scheduler = lr_scheduler.CosineAnnealingLR(teacher_optim, T_max=num_epochs, eta_min=1e-6)

set_seed(42)
student = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', weights=None)
student.fc = nn.Linear(student.fc.in_features, 10)
student.to(device)
student_optim = torch.optim.Adam(student.parameters(), lr=1e-3)
student_scheduler = lr_scheduler.CosineAnnealingLR(student_optim, T_max=num_epochs, eta_min=1e-6)


train_val_loop(teacher, teacher_optim, num_epochs, vanilla_train_epoch, teacher_scheduler, print_stats=True)
train_val_loop(student, student_optim, num_epochs, vanilla_train_epoch, student_scheduler, print_stats=True)

Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


Epoch  Train Loss  Train Acc  Val Loss  Val Acc
----------------------------------------------------


Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


    1      0.9988     0.6570    0.7590   0.7493
    2      0.7045     0.7603    0.6194   0.7917
    3      0.5573     0.8087    0.5355   0.8148
    4      0.4474     0.8465    0.4660   0.8391
    5      0.3655     0.8734    0.3972   0.8635
Epoch  Train Loss  Train Acc  Val Loss  Val Acc
----------------------------------------------------
    1      1.5249     0.4454    1.2468   0.5607
    2      1.1542     0.5876    1.0171   0.6346
    3      0.9690     0.6586    0.8527   0.6993
    4      0.8357     0.7074    0.7905   0.7241
    5      0.7308     0.7423    0.7060   0.7587


In [ ]:
teacher_params = "{:,}".format(sum(p.numel() for p in teacher.parameters()))
print(f"resnet50 parameters: {teacher_params}")
student_params = "{:,}".format(sum(p.numel() for p in student.parameters()))
print(f"resnet18 parameters: {student_params}")

DeepNN parameters: 23,528,522
LightNN parameters: 11,181,642


In [13]:
kl_div = nn.KLDivLoss(reduction='batchmean')

def distill_loss(student_logits, teacher_logits, labels, criterion, T, alpha):
    log_softmax_student = F.log_softmax(student_logits / T, dim=1)
    softmax_teacher = F.softmax(teacher_logits / T, dim=1)
    hard_loss = criterion(student_logits, labels)
    soft_loss = T **2 * kl_div(log_softmax_student, softmax_teacher)
    return (1.0 - alpha) * hard_loss + alpha * soft_loss

In [14]:
def distill_training(student, loader, student_optim, criterion, scaler, teacher, T, alpha): 
    student.train()
    teacher.eval()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        student_optim.zero_grad()
        with torch.amp.autocast('cuda'):
            student_logits = student(images)
            with torch.no_grad():
                teacher_logits = teacher(images)
            loss = distill_loss(student_logits, teacher_logits, labels, criterion, T, alpha)

        scaler.scale(loss).backward()
        scaler.step(student_optim)
        scaler.update()
        total_loss += loss.item()
        correct += (student_logits.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

In [15]:
set_seed(42)
dist_student = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', weights=None)
dist_student.fc = nn.Linear(dist_student.fc.in_features, 10)
dist_student.to(device)
dist_student_optim = torch.optim.Adam(dist_student.parameters(), lr=1e-3)
dist_student_scheduler = lr_scheduler.CosineAnnealingLR(dist_student_optim, T_max=num_epochs, eta_min=1e-6)


train_val_loop(dist_student, dist_student_optim, num_epochs, distill_training, dist_student_scheduler, teacher=teacher, T=1.5, alpha=0.7)

Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


Epoch  Train Loss  Train Acc  Val Loss  Val Acc
----------------------------------------------------
    1      1.8719     0.4696    1.3577   0.5525
    2      1.2476     0.6176    0.9684   0.6722
    3      0.9807     0.6845    0.8650   0.7135
    4      0.7887     0.7289    0.7683   0.7416
    5      0.6436     0.7564    0.6928   0.7628


In [ ]:
def train_val_loop_early_stopping(model, optimizer, epochs, training_function, lr_scheduler, seed=42,
                    print_stats=True, patience=3, min_delta=1e-4, **kwargs):
    trainloader, testloader = get_dataloaders(seed)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')

    if print_stats:
        print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>8}  {'Val Acc':>7}")
        print("-" * 52)

    best_val_loss = float('inf')
    best_val_acc = 0.0
    epochs_no_improve = 0

    for epoch in range(epochs):
        tr_loss, tr_acc = training_function(model, trainloader, optimizer, criterion, scaler, **kwargs)
        vl_loss, vl_acc = evaluate(model, testloader, criterion)

        if print_stats:
            print(f"{epoch+1:>5}  {tr_loss:>10.4f}  {tr_acc:>9.4f}  {vl_loss:>8.4f}  {vl_acc:>7.4f}")

        if vl_loss < best_val_loss - min_delta:
            best_val_loss = vl_loss
            best_val_acc = vl_acc
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")


        lr_scheduler.step()

    return best_val_loss, best_val_acc, epoch+1

In [19]:
import itertools

T_values = [1.5, 1.75, 2.0, 2.5]
alpha_values = [0.4, 0.5, 0.6]

results = []

for T, alpha in itertools.product(T_values, alpha_values):
    set_seed(42)
    dist_student = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', weights=None)
    dist_student.fc = nn.Linear(dist_student.fc.in_features, 10)
    dist_student.to(device)
    dist_student_optim = torch.optim.Adam(dist_student.parameters(), lr=1e-3)
    dist_student_scheduler = lr_scheduler.CosineAnnealingLR(dist_student_optim, T_max=num_epochs, eta_min=1e-6)

    val_loss, val_acc = train_val_loop_early_stopping(
        dist_student, dist_student_optim, num_epochs, distill_training,
        dist_student_scheduler, print_stats=False,
        teacher=teacher, T=T, alpha=alpha
    )

    print(f"T={T:>4}  alpha={alpha:>4}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")
    results.append({'T': T, 'alpha': alpha, 'val_loss': val_loss, 'val_acc': val_acc})

# Summary, best first
results.sort(key=lambda r: r['val_acc'], reverse=True)
print(f"\n{'T':>5}  {'alpha':>6}  {'Val Loss':>9}  {'Val Acc':>8}")
print("-" * 35)
for r in results:
    print(f"{r['T']:>5}  {r['alpha']:>6}  {r['val_loss']:>9.4f}  {r['val_acc']:>8.4f}")

best = results[0]
print(f"\nBest config: T={best['T']}, alpha={best['alpha']}, val_acc={best['val_acc']:.4f}")

Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


T= 1.5  alpha= 0.4  val_loss=0.6839  val_acc=0.7643


Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


T= 1.5  alpha= 0.5  val_loss=0.6831  val_acc=0.7647


Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


...

T= 2.5  alpha= 0.4  val_loss=0.6902  val_acc=0.7677


Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


T= 2.5  alpha= 0.5  val_loss=0.6961  val_acc=0.7706


Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


T= 2.5  alpha= 0.6  val_loss=0.7066  val_acc=0.7671

    T   alpha   Val Loss   Val Acc
-----------------------------------
 1.75     0.5     0.6749    0.7736
  2.5     0.5     0.6961    0.7706
 1.75     0.4     0.6798    0.7684
  2.0     0.5     0.6867    0.7683
  2.0     0.4     0.6815    0.7681
  2.5     0.4     0.6902    0.7677
  2.5     0.6     0.7066    0.7671
  1.5     0.6     0.6936    0.7663
  2.0     0.6     0.7040    0.7650
  1.5     0.5     0.6831    0.7647
  1.5     0.4     0.6839    0.7643
 1.75     0.6     0.7014    0.7623

Best config: T=1.75, alpha=0.5, val_acc=0.7736


In [21]:
import itertools

T_values = [1.75, 2.0, 2.25 ,2.5]
alpha_values = [0.4, 0.45, 0.5]

results = []

for T, alpha in itertools.product(T_values, alpha_values):
    set_seed(42)
    dist_student = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', weights=None)
    dist_student.fc = nn.Linear(dist_student.fc.in_features, 10)
    dist_student.to(device)
    dist_student_optim = torch.optim.Adam(dist_student.parameters(), lr=1e-3)
    dist_student_scheduler = lr_scheduler.CosineAnnealingLR(dist_student_optim, T_max=num_epochs, eta_min=1e-6)

    val_loss, val_acc = train_val_loop_early_stopping(
        dist_student, dist_student_optim, num_epochs, distill_training,
        dist_student_scheduler, print_stats=False,
        teacher=teacher, T=T, alpha=alpha
    )

    print(f"T={T:>4}  alpha={alpha:>4}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")
    results.append({'T': T, 'alpha': alpha, 'val_loss': val_loss, 'val_acc': val_acc})

# Summary, best first
results.sort(key=lambda r: r['val_acc'], reverse=True)
print(f"\n{'T':>5}  {'alpha':>6}  {'Val Loss':>9}  {'Val Acc':>8}")
print("-" * 35)
for r in results:
    print(f"{r['T']:>5}  {r['alpha']:>6}  {r['val_loss']:>9.4f}  {r['val_acc']:>8.4f}")

best = results[0]
print(f"\nBest config: T={best['T']}, alpha={best['alpha']}, val_acc={best['val_acc']:.4f}")

Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


T=1.75  alpha= 0.4  val_loss=0.6798  val_acc=0.7684


Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


T=1.75  alpha=0.45  val_loss=0.6958  val_acc=0.7598


Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


...

T= 2.5  alpha= 0.4  val_loss=0.6902  val_acc=0.7677


Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


T= 2.5  alpha=0.45  val_loss=0.7031  val_acc=0.7697


Using cache found in /home/sara/.cache/torch/hub/pytorch_vision_v0.10.0


T= 2.5  alpha= 0.5  val_loss=0.6961  val_acc=0.7706

    T   alpha   Val Loss   Val Acc
-----------------------------------
 1.75     0.5     0.6749    0.7736
  2.5     0.5     0.6961    0.7706
  2.5    0.45     0.7031    0.7697
 2.25     0.5     0.6966    0.7687
 1.75     0.4     0.6798    0.7684
  2.0     0.5     0.6867    0.7683
  2.0     0.4     0.6815    0.7681
  2.5     0.4     0.6902    0.7677
  2.0    0.45     0.7024    0.7666
 2.25     0.4     0.7036    0.7658
 2.25    0.45     0.7041    0.7645
 1.75    0.45     0.6958    0.7598

Best config: T=1.75, alpha=0.5, val_acc=0.7736
